# 北海道紋別市・湧別町 土地利用分類 2020年版

Landsat 8 Collection 2 SR を使用したRandom Forest分類

## 概要
- 農地（牧草地・飼料用トウモロコシ畑・畑作農地）・森林・市街地・水域等の識別
- 5-9月の時系列コンポジットを使用
- Random Forestで植生指数の時系列統計と地形データを特徴量として使用

## クラス定義
| ID | クラス | 説明 |
|----|-------|------|
| 1 | 牧草地 | 牧草・採草地 |
| 2 | トウモロコシ畑 | 飼料用トウモロコシ |
| 3 | 畑作農地 | 小麦・てんさい等 |
| 4 | 森林 | 針葉樹・広葉樹 |
| 5 | 市街地 | 建物・道路 |
| 6 | 水域 | 河川・湖沼 |
| 7 | 裸地 | 裸地・その他 |

---
# 1. 環境セットアップ

In [ ]:
# 必要なパッケージをインストール
!pip install -q earthengine-api geopandas folium geemap

In [ ]:
# ライブラリのインポート
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import json
from google.colab import files

## 1.1 Google Earth Engine 認証・初期化

⚠️ **重要**: `YOUR_PROJECT_ID` を自分のプロジェクトIDに置き換えてください

プロジェクトIDの確認方法:
1. https://code.earthengine.google.com/ にアクセス
2. 右上のアカウント → Cloud Project を確認
3. または https://console.cloud.google.com/ でプロジェクト作成

In [ ]:
# ============================================
# ★★★ ここにプロジェクトIDを入力 ★★★
# ============================================
PROJECT_ID = 'YOUR_PROJECT_ID'  # 例: 'ee-username' や 'my-gee-project-12345'

# 認証（ブラウザが開きます）
ee.Authenticate()

# 初期化
ee.Initialize(project=PROJECT_ID)

print('✅ Google Earth Engine 初期化成功！')

---
# 2. 設定パラメータ

In [ ]:
# ============================================
# 対象地域の設定（紋別市・湧別町の範囲）
# ============================================
STUDY_AREA = {
    'min_lon': 143.0,
    'max_lon': 144.5,
    'min_lat': 43.8,
    'max_lat': 44.6
}

# 分類カテゴリ
LAND_USE_CLASSES = {
    1: 'grassland',           # 牧草地
    2: 'corn_field',          # 飼料用トウモロコシ畑
    3: 'other_cropland',      # 畑作農地（その他）
    4: 'forest',              # 森林
    5: 'urban',               # 市街地
    6: 'water',               # 水域
    7: 'bare_land'            # 裸地・その他
}

# 分析対象年と期間
TARGET_YEAR = 2020
START_DATE = f'{TARGET_YEAR}-05-01'
END_DATE = f'{TARGET_YEAR}-09-30'

# 雲量閾値
CLOUD_COVER_MAX = 20

# Random Forest パラメータ
RF_PARAMS = {
    'numberOfTrees': 100,
    'minLeafPopulation': 5,
    'bagFraction': 0.7,
    'seed': 42
}

print('設定完了')
print(f'対象年: {TARGET_YEAR}')
print(f'対象期間: {START_DATE} ～ {END_DATE}')

---
# 3. 対象地域の定義と確認

In [ ]:
# 対象地域のジオメトリを作成
geometry = ee.Geometry.Rectangle([
    STUDY_AREA['min_lon'], STUDY_AREA['min_lat'],
    STUDY_AREA['max_lon'], STUDY_AREA['max_lat']
])

# 地図で確認
Map = geemap.Map(center=[44.2, 143.7], zoom=9)
Map.addLayer(geometry, {'color': 'blue'}, '対象地域')
Map

---
# 4. Landsat 8 データ取得・前処理

In [ ]:
def mask_landsat8_clouds(image):
    """
    Landsat 8 Collection 2 SR の雲マスキング
    QA_PIXELバンドを使用
    """
    qa = image.select('QA_PIXEL')
    
    # Bit 3: Cloud, Bit 4: Cloud Shadow
    cloud_bit_mask = 1 << 3
    cloud_shadow_bit_mask = 1 << 4
    
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0) \
        .And(qa.bitwiseAnd(cloud_shadow_bit_mask).eq(0))
    
    return image.updateMask(mask)


def apply_scale_factors(image):
    """
    Landsat 8 Collection 2 SR のスケールファクター適用
    """
    optical_bands = image.select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']) \
        .multiply(0.0000275).add(-0.2)
    
    return image.addBands(optical_bands, overwrite=True)


# Landsat 8 コレクションを取得
collection = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
    .filterBounds(geometry) \
    .filterDate(START_DATE, END_DATE) \
    .filter(ee.Filter.lt('CLOUD_COVER', CLOUD_COVER_MAX)) \
    .map(mask_landsat8_clouds) \
    .map(apply_scale_factors)

image_count = collection.size().getInfo()
print(f'✅ 取得画像数: {image_count}')

In [ ]:
# 取得した画像の一覧を表示
image_list = collection.aggregate_array('system:index').getInfo()
print('取得した画像:')
for img_id in image_list[:10]:  # 最初の10枚
    print(f'  - {img_id}')
if len(image_list) > 10:
    print(f'  ... 他 {len(image_list) - 10} 枚')

---
# 5. 植生指数の計算

In [ ]:
def add_vegetation_indices(image):
    """
    植生指数を計算して追加
    
    Landsat 8 バンド対応:
    - Blue: SR_B2
    - Green: SR_B3
    - Red: SR_B4
    - NIR: SR_B5
    - SWIR1: SR_B6
    - SWIR2: SR_B7
    """
    # NDVI: (NIR - Red) / (NIR + Red)
    ndvi = image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    
    # EVI: 2.5 * (NIR - Red) / (NIR + 6*Red - 7.5*Blue + 1)
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))',
        {
            'NIR': image.select('SR_B5'),
            'RED': image.select('SR_B4'),
            'BLUE': image.select('SR_B2')
        }
    ).rename('EVI')
    
    # NDWI: (Green - NIR) / (Green + NIR) - 水域検出
    ndwi = image.normalizedDifference(['SR_B3', 'SR_B5']).rename('NDWI')
    
    # LSWI: (NIR - SWIR1) / (NIR + SWIR1) - 植生水分
    lswi = image.normalizedDifference(['SR_B5', 'SR_B6']).rename('LSWI')
    
    # NDBI: (SWIR1 - NIR) / (SWIR1 + NIR) - 市街地検出
    ndbi = image.normalizedDifference(['SR_B6', 'SR_B5']).rename('NDBI')
    
    return image.addBands([ndvi, evi, ndwi, lswi, ndbi])

print('✅ 植生指数計算関数を定義')

In [ ]:
# 植生指数を可視化（サンプル画像）
sample_image = collection.median()
sample_with_indices = add_vegetation_indices(sample_image)

Map2 = geemap.Map(center=[44.2, 143.7], zoom=10)

# True Color
vis_tc = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}
Map2.addLayer(sample_with_indices.clip(geometry), vis_tc, 'True Color')

# NDVI
vis_ndvi = {'min': -0.2, 'max': 0.8, 'palette': ['red', 'yellow', 'green', 'darkgreen']}
Map2.addLayer(sample_with_indices.select('NDVI').clip(geometry), vis_ndvi, 'NDVI')

Map2.addLayerControl()
Map2

---
# 6. 月別コンポジット作成

In [ ]:
def create_monthly_composites(collection, year):
    """
    5-9月の月別中央値コンポジットを作成
    """
    months = [5, 6, 7, 8, 9]
    composites = []
    
    for month in months:
        start = f'{year}-{month:02d}-01'
        if month == 9:
            end = f'{year}-{month:02d}-30'
        else:
            end = f'{year}-{month+1:02d}-01'
        
        monthly_col = collection.filterDate(start, end)
        composite = monthly_col.median().set('month', month)
        composite = add_vegetation_indices(composite)
        
        # バンド名にプレフィックスを追加
        bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7',
                 'NDVI', 'EVI', 'NDWI', 'LSWI', 'NDBI']
        renamed_bands = [f'M{month}_{b}' for b in bands]
        
        composite = composite.select(bands).rename(renamed_bands)
        composites.append(composite)
    
    # 全月をスタック
    stacked = composites[0]
    for comp in composites[1:]:
        stacked = stacked.addBands(comp)
    
    return stacked

monthly_composites = create_monthly_composites(collection, TARGET_YEAR)
print('✅ 月別コンポジット作成完了')
print(f'バンド数: {len(monthly_composites.bandNames().getInfo())}')

---
# 7. 時系列統計量の計算

In [ ]:
def calculate_temporal_statistics(collection):
    """
    植生指数の時系列統計量を計算
    """
    collection_with_indices = collection.map(add_vegetation_indices)
    
    indices = ['NDVI', 'EVI', 'NDWI', 'LSWI']
    stats_image = ee.Image()
    
    for idx in indices:
        idx_collection = collection_with_indices.select(idx)
        
        max_val = idx_collection.max().rename(f'{idx}_max')
        min_val = idx_collection.min().rename(f'{idx}_min')
        mean_val = idx_collection.mean().rename(f'{idx}_mean')
        std_val = idx_collection.reduce(ee.Reducer.stdDev()).rename(f'{idx}_std')
        cv = std_val.divide(mean_val.abs().add(0.001)).rename(f'{idx}_cv')
        
        stats_image = stats_image.addBands([max_val, min_val, mean_val, std_val, cv])
    
    return stats_image

temporal_stats = calculate_temporal_statistics(collection)
print('✅ 時系列統計量計算完了')

---
# 8. フェノロジー特徴量（牧草地 vs トウモロコシ識別用）

In [ ]:
def calculate_phenology_features(collection, year):
    """
    フェノロジー（生育段階）特徴量を計算
    牧草地 vs トウモロコシ識別に重要
    
    ★ポイント★
    - トウモロコシは7-8月にNDVIが急上昇（牧草地より高い）
    - 牧草地は5-9月通じて比較的安定したNDVI
    """
    july_col = collection.filterDate(f'{year}-07-01', f'{year}-07-31').map(add_vegetation_indices)
    aug_col = collection.filterDate(f'{year}-08-01', f'{year}-08-31').map(add_vegetation_indices)
    sept_col = collection.filterDate(f'{year}-09-01', f'{year}-09-30').map(add_vegetation_indices)
    
    july_ndvi = july_col.select('NDVI').median().rename('NDVI_july')
    aug_ndvi = aug_col.select('NDVI').median().rename('NDVI_aug')
    sept_ndvi = sept_col.select('NDVI').median().rename('NDVI_sept')
    
    # 8月-7月のNDVI差分（トウモロコシで大きい）
    ndvi_diff_aug_july = aug_ndvi.subtract(july_ndvi).rename('NDVI_diff_aug_july')
    
    # 9月-8月のNDVI差分（トウモロコシで負に大きい＝収穫）
    ndvi_diff_sept_aug = sept_ndvi.subtract(aug_ndvi).rename('NDVI_diff_sept_aug')
    
    # 夏季（7-8月）のNDVI最大値
    summer_max_ndvi = july_col.merge(aug_col).select('NDVI').max().rename('NDVI_summer_max')
    
    phenology = ee.Image.cat([
        july_ndvi, aug_ndvi, sept_ndvi,
        ndvi_diff_aug_july, ndvi_diff_sept_aug,
        summer_max_ndvi
    ])
    
    return phenology

phenology = calculate_phenology_features(collection, TARGET_YEAR)
print('✅ フェノロジー特徴量計算完了')

In [ ]:
# 8月NDVIの可視化（トウモロコシ識別に重要）
Map3 = geemap.Map(center=[44.2, 143.7], zoom=11)

vis_ndvi = {'min': 0.3, 'max': 0.9, 'palette': ['yellow', 'lightgreen', 'green', 'darkgreen']}
Map3.addLayer(phenology.select('NDVI_aug').clip(geometry), vis_ndvi, '8月 NDVI（トウモロコシ＝高い）')

Map3

---
# 9. 地形データの追加

In [ ]:
def add_terrain_features():
    """
    SRTM DEMから地形特徴量を計算
    """
    dem = ee.Image('USGS/SRTMGL1_003')
    
    elevation = dem.select('elevation').rename('elevation')
    slope = ee.Terrain.slope(dem).rename('slope')
    aspect = ee.Terrain.aspect(dem).rename('aspect')
    
    # 方位を sin/cos に変換（連続値として扱う）
    aspect_sin = aspect.multiply(3.14159 / 180).sin().rename('aspect_sin')
    aspect_cos = aspect.multiply(3.14159 / 180).cos().rename('aspect_cos')
    
    terrain = ee.Image.cat([elevation, slope, aspect_sin, aspect_cos])
    
    return terrain

terrain = add_terrain_features()
print('✅ 地形データ追加完了')

---
# 10. 特徴量スタックの作成

In [ ]:
# 全特徴量をスタック
feature_stack = ee.Image.cat([
    monthly_composites,
    temporal_stats,
    phenology,
    terrain
]).clip(geometry)

band_names = feature_stack.bandNames().getInfo()
print(f'✅ 特徴量スタック作成完了')
print(f'特徴量数: {len(band_names)}')
print(f'\n特徴量一覧:')
for i, name in enumerate(band_names):
    print(f'  {i+1:2d}. {name}')

---
# 11. 教師データの準備

## ⚠️ 重要: 教師データを準備してください

以下の3つの方法から選択:

### 方法A: GeoJSONファイルをアップロード
### 方法B: GEE Assetから読み込み
### 方法C: 手動でポイントを定義（テスト用）

In [ ]:
# ============================================
# 方法A: GeoJSONファイルをアップロード
# ============================================
# 以下のコメントを解除して使用

# print('GeoJSONファイルをアップロードしてください')
# uploaded = files.upload()
# 
# # アップロードしたファイルを読み込み
# filename = list(uploaded.keys())[0]
# gdf = gpd.read_file(filename)
# 
# # GeoJSONをGEE FeatureCollectionに変換
# features = []
# for idx, row in gdf.iterrows():
#     geom = row.geometry.__geo_interface__
#     props = {'class': int(row.get('class', row.get('landuse', 0)))}
#     feature = ee.Feature(ee.Geometry(geom), props)
#     features.append(feature)
# 
# training_data = ee.FeatureCollection(features)
# print(f'✅ 教師データ読み込み完了: {len(features)} サンプル')

In [ ]:
# ============================================
# 方法B: GEE Asset から読み込み
# ============================================
# 以下のコメントを解除して使用（asset_idを変更）

# asset_id = 'users/YOUR_USERNAME/training_data_2020'
# training_data = ee.FeatureCollection(asset_id)
# print(f'✅ 教師データ読み込み完了: {training_data.size().getInfo()} サンプル')

In [ ]:
# ============================================
# 方法C: 手動でポイントを定義（テスト用）
# ============================================
# ★ 実際の分析では正確な教師データを使用してください ★

# サンプルポイント（座標は実際のデータに置き換えが必要）
training_data = ee.FeatureCollection([
    # 牧草地 (class=1)
    ee.Feature(ee.Geometry.Point([143.45, 44.18]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([143.50, 44.22]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([143.55, 44.25]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([143.48, 44.15]), {'class': 1}),
    ee.Feature(ee.Geometry.Point([143.52, 44.20]), {'class': 1}),
    
    # トウモロコシ畑 (class=2)
    ee.Feature(ee.Geometry.Point([143.65, 44.12]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([143.70, 44.15]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([143.68, 44.10]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([143.72, 44.18]), {'class': 2}),
    ee.Feature(ee.Geometry.Point([143.67, 44.14]), {'class': 2}),
    
    # 畑作農地 (class=3)
    ee.Feature(ee.Geometry.Point([143.60, 44.08]), {'class': 3}),
    ee.Feature(ee.Geometry.Point([143.62, 44.12]), {'class': 3}),
    ee.Feature(ee.Geometry.Point([143.58, 44.10]), {'class': 3}),
    
    # 森林 (class=4)
    ee.Feature(ee.Geometry.Point([143.35, 44.40]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([143.40, 44.45]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([143.38, 44.42]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([143.42, 44.48]), {'class': 4}),
    ee.Feature(ee.Geometry.Point([143.45, 44.50]), {'class': 4}),
    
    # 市街地 (class=5)
    ee.Feature(ee.Geometry.Point([143.35, 44.35]), {'class': 5}),
    ee.Feature(ee.Geometry.Point([143.36, 44.34]), {'class': 5}),
    ee.Feature(ee.Geometry.Point([143.34, 44.36]), {'class': 5}),
    
    # 水域 (class=6)
    ee.Feature(ee.Geometry.Point([143.90, 44.00]), {'class': 6}),
    ee.Feature(ee.Geometry.Point([143.88, 44.02]), {'class': 6}),
    ee.Feature(ee.Geometry.Point([143.92, 43.98]), {'class': 6}),
    
    # 裸地 (class=7)
    ee.Feature(ee.Geometry.Point([143.80, 44.05]), {'class': 7}),
    ee.Feature(ee.Geometry.Point([143.82, 44.03]), {'class': 7}),
])

print('⚠️ サンプル教師データを使用中')
print('   実際の分析では正確な教師データを使用してください')
print(f'\nサンプル数: {training_data.size().getInfo()}')

In [ ]:
# 教師データを地図で確認
Map4 = geemap.Map(center=[44.2, 143.7], zoom=10)

# 背景にTrue Color
vis_tc = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.3}
Map4.addLayer(sample_with_indices.clip(geometry), vis_tc, 'True Color')

# 教師データをクラス別に色分け
class_colors = {
    1: 'lightgreen',  # 牧草地
    2: 'yellow',      # トウモロコシ
    3: 'orange',      # 畑作
    4: 'darkgreen',   # 森林
    5: 'red',         # 市街地
    6: 'blue',        # 水域
    7: 'gray'         # 裸地
}

Map4.addLayer(training_data, {'color': 'white'}, '教師データ')
Map4

---
# 12. Random Forest 分類

In [ ]:
# 教師データにバンド値を抽出
training_samples = feature_stack.sampleRegions(
    collection=training_data,
    properties=['class'],
    scale=30,
    tileScale=8
)

print(f'訓練サンプル数: {training_samples.size().getInfo()}')

In [ ]:
# Random Forest 分類器を訓練
classifier = ee.Classifier.smileRandomForest(**RF_PARAMS).train(
    features=training_samples,
    classProperty='class',
    inputProperties=band_names
)

print('✅ Random Forest 訓練完了')

In [ ]:
# 分類実行
classified = feature_stack.classify(classifier)

print('✅ 分類完了')

In [ ]:
# 分類結果を表示
Map5 = geemap.Map(center=[44.2, 143.7], zoom=10)

# 分類結果の色設定
class_palette = [
    '#90EE90',  # 1: 牧草地 - ライトグリーン
    '#FFD700',  # 2: トウモロコシ - ゴールド
    '#FFA500',  # 3: 畑作 - オレンジ
    '#006400',  # 4: 森林 - ダークグリーン
    '#FF0000',  # 5: 市街地 - 赤
    '#0000FF',  # 6: 水域 - 青
    '#808080',  # 7: 裸地 - グレー
]

vis_class = {'min': 1, 'max': 7, 'palette': class_palette}
Map5.addLayer(classified.clip(geometry), vis_class, '土地利用分類')

# 凡例を追加
legend_dict = {
    '1: 牧草地': '#90EE90',
    '2: トウモロコシ畑': '#FFD700',
    '3: 畑作農地': '#FFA500',
    '4: 森林': '#006400',
    '5: 市街地': '#FF0000',
    '6: 水域': '#0000FF',
    '7: 裸地': '#808080',
}
Map5.add_legend(title='土地利用', legend_dict=legend_dict)

Map5

---
# 13. 精度評価

In [ ]:
# データを訓練セットとテストセットに分割
training_data_with_random = training_data.randomColumn('random')
train_set = training_data_with_random.filter(ee.Filter.lt('random', 0.7))
test_set = training_data_with_random.filter(ee.Filter.gte('random', 0.7))

print(f'訓練セット: {train_set.size().getInfo()} サンプル')
print(f'テストセット: {test_set.size().getInfo()} サンプル')

In [ ]:
# 訓練セットでモデル再訓練
train_samples = feature_stack.sampleRegions(
    collection=train_set,
    properties=['class'],
    scale=30,
    tileScale=8
)

classifier_eval = ee.Classifier.smileRandomForest(**RF_PARAMS).train(
    features=train_samples,
    classProperty='class',
    inputProperties=band_names
)

# テストセットで評価
test_samples = feature_stack.sampleRegions(
    collection=test_set,
    properties=['class'],
    scale=30,
    tileScale=8
)

validated = test_samples.classify(classifier_eval)

In [ ]:
# 混同行列と精度指標
confusion_matrix = validated.errorMatrix('class', 'classification')

print('=' * 50)
print('精度評価結果')
print('=' * 50)

overall_accuracy = confusion_matrix.accuracy().getInfo()
kappa = confusion_matrix.kappa().getInfo()

print(f'\n全体精度 (Overall Accuracy): {overall_accuracy:.4f}')
print(f'カッパ係数 (Kappa): {kappa:.4f}')

print('\n混同行列:')
cm_array = confusion_matrix.getInfo()
print(np.array(cm_array))

---
# 14. 特徴量重要度

In [ ]:
# 特徴量重要度を取得
importance = classifier.explain().getInfo()

if 'importance' in importance:
    imp_dict = importance['importance']
    
    # DataFrameに変換
    importance_df = pd.DataFrame({
        'feature': list(imp_dict.keys()),
        'importance': list(imp_dict.values())
    }).sort_values('importance', ascending=False)
    
    print('\n上位20の重要な特徴量:')
    print('=' * 40)
    for i, row in importance_df.head(20).iterrows():
        print(f"{row['feature']:30s} {row['importance']:.4f}")
else:
    print('特徴量重要度を取得できませんでした')

---
# 15. 結果のエクスポート

In [ ]:
# Google Drive にエクスポート
export_task = ee.batch.Export.image.toDrive(
    image=classified.toInt8(),
    description='Mombetsu_LandUse_2020',
    folder='LandUse_Mombetsu',
    region=geometry,
    scale=30,
    maxPixels=1e13,
    crs='EPSG:32654'  # UTM Zone 54N
)

export_task.start()

print('✅ エクスポートタスク開始')
print(f'タスクID: {export_task.id}')
print('\nGoogle Drive の "LandUse_Mombetsu" フォルダに保存されます')
print('タスクの進行状況は GEE Code Editor の Tasks タブで確認できます')

In [ ]:
# タスクの状態を確認
import time

print('エクスポート状態を確認中...')
while True:
    status = export_task.status()
    state = status['state']
    print(f'状態: {state}')
    
    if state in ['COMPLETED', 'FAILED', 'CANCELLED']:
        break
    
    time.sleep(30)  # 30秒ごとに確認

if state == 'COMPLETED':
    print('\n✅ エクスポート完了！')
else:
    print(f'\n❌ エクスポート失敗: {status}')

---
# 16. まとめ

## 実行結果
- 使用データ: Landsat 8 Collection 2 SR
- 対象期間: 2020年5-9月
- 分類手法: Random Forest
- 特徴量: 月別コンポジット、時系列統計、フェノロジー、地形

## 精度向上のためのヒント
1. **教師データの品質向上**: 実際の現地調査データや航空写真を使用
2. **サンプル数の増加**: 各クラス50-100サンプル以上を推奨
3. **空間分散**: 対象地域全体に均等にサンプルを配置

## 牧草地 vs トウモロコシの識別ポイント
- **8月のNDVI** が最も重要な特徴量
- トウモロコシ: 8月にNDVI > 0.8
- 牧草地: 5-9月通じてNDVI 0.5-0.7で安定